In [1]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from datetime import datetime

batch_id = datetime.now().strftime("%Y%m%d%H%M%S")

for schema_name in ["bronze", "silver", "gold", "quarantine"]:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {schema_name}")

print(f"Gold fact build started. Batch ID: {batch_id}")

StatementMeta(, b01a3b81-37fc-47f7-9f94-230208af0de3, 3, Finished, Available, Finished, False)

Gold fact build started. Batch ID: 20260715004706


In [2]:
def write_delta(df, table_name):
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(table_name)
    )
    print(f"{table_name}: {df.count()} rows")

StatementMeta(, b01a3b81-37fc-47f7-9f94-230208af0de3, 4, Finished, Available, Finished, False)

In [3]:
stores = spark.table("silver.stores")

store_window = Window.orderBy("store_id")

dim_store = (
    stores
    .select(
        "store_id",
        "store_name",
        "region",
        "city",
        "country",
        "store_type",
        "open_date"
    )
    .dropDuplicates(["store_id"])
    .withColumn("store_sk", F.row_number().over(store_window))
    .select(
        "store_sk",
        "store_id",
        "store_name",
        "region",
        "city",
        "country",
        "store_type",
        "open_date"
    )
)

write_delta(dim_store, "gold.dim_store")

StatementMeta(, b01a3b81-37fc-47f7-9f94-230208af0de3, 5, Finished, Available, Finished, False)

gold.dim_store: 8 rows


In [4]:
channels = (
    spark.table("silver.orders")
    .select("channel")
    .where(F.col("channel").isNotNull())
    .dropDuplicates()
)

channel_window = Window.orderBy("channel")

dim_channel = (
    channels
    .withColumn("channel_sk", F.row_number().over(channel_window))
    .select(
        "channel_sk",
        "channel"
    )
)

write_delta(dim_channel, "gold.dim_channel")

StatementMeta(, b01a3b81-37fc-47f7-9f94-230208af0de3, 6, Finished, Available, Finished, False)

gold.dim_channel: 3 rows


In [5]:
orders_dates = (
    spark.table("silver.orders")
    .select(F.to_date("order_date").alias("calendar_date"))
)

returns_dates = (
    spark.table("silver.returns")
    .select(F.to_date("return_date").alias("calendar_date"))
)

inventory_dates = (
    spark.table("silver.inventory")
    .select(F.to_date("snapshot_date").alias("calendar_date"))
)

all_dates = (
    orders_dates
    .union(returns_dates)
    .union(inventory_dates)
    .where(F.col("calendar_date").isNotNull())
    .dropDuplicates()
)

dim_date = (
    all_dates
    .withColumn("date_sk", F.date_format("calendar_date", "yyyyMMdd").cast("int"))
    .withColumn("year", F.year("calendar_date"))
    .withColumn("quarter", F.quarter("calendar_date"))
    .withColumn("month", F.month("calendar_date"))
    .withColumn("month_name", F.date_format("calendar_date", "MMMM"))
    .withColumn("day_of_month", F.dayofmonth("calendar_date"))
    .withColumn("day_of_week", F.dayofweek("calendar_date"))
    .withColumn("day_name", F.date_format("calendar_date", "EEEE"))
    .withColumn("week_of_year", F.weekofyear("calendar_date"))
    .withColumn(
        "is_weekend",
        F.when(F.dayofweek("calendar_date").isin(1, 7), F.lit(True)).otherwise(F.lit(False))
    )
    .select(
        "date_sk",
        "calendar_date",
        "year",
        "quarter",
        "month",
        "month_name",
        "day_of_month",
        "day_of_week",
        "day_name",
        "week_of_year",
        "is_weekend"
    )
)

write_delta(dim_date, "gold.dim_date")

StatementMeta(, b01a3b81-37fc-47f7-9f94-230208af0de3, 7, Finished, Available, Finished, False)

gold.dim_date: 116 rows


In [6]:
orders = spark.table("silver.orders").alias("o")
dim_customer = spark.table("gold.dim_customer").alias("dc")
dim_product = spark.table("gold.dim_product").alias("dp")
dim_store = spark.table("gold.dim_store").alias("ds")
dim_channel = spark.table("gold.dim_channel").alias("dch")
dim_date = spark.table("gold.dim_date").alias("dd")

orders_prepared = (
    orders
    .withColumn("order_calendar_date", F.to_date("order_date"))
    .withColumn("order_date_sk", F.date_format(F.to_date("order_date"), "yyyyMMdd").cast("int"))
)

fact_sales = (
    orders_prepared.alias("o")
    .join(
        dim_customer.alias("dc"),
        (
            (F.col("o.customer_id") == F.col("dc.customer_id")) &
            (F.col("o.order_calendar_date") >= F.col("dc.effective_start_date")) &
            (F.col("o.order_calendar_date") <= F.col("dc.effective_end_date"))
        ),
        "left"
    )
    .join(
        dim_product.alias("dp"),
        (
            (F.col("o.product_id") == F.col("dp.product_id")) &
            (F.col("o.order_calendar_date") >= F.col("dp.effective_start_date")) &
            (F.col("o.order_calendar_date") <= F.col("dp.effective_end_date"))
        ),
        "left"
    )
    .join(
        dim_store.alias("ds"),
        F.col("o.store_id") == F.col("ds.store_id"),
        "left"
    )
    .join(
        dim_channel.alias("dch"),
        F.col("o.channel") == F.col("dch.channel"),
        "left"
    )
    .join(
        dim_date.alias("dd"),
        F.col("o.order_date_sk") == F.col("dd.date_sk"),
        "left"
    )
    .withColumn("gross_sales_amount", F.col("o.quantity") * F.col("o.unit_price"))
    .withColumn(
        "net_sales_amount",
        F.when(
            F.col("o.order_status") == "Cancelled",
            F.lit(0.0)
        ).otherwise(
            (F.col("o.quantity") * F.col("o.unit_price")) - F.col("o.discount_amount")
        )
    )
    .withColumn(
        "is_cancelled",
        F.when(F.col("o.order_status") == "Cancelled", F.lit(True)).otherwise(F.lit(False))
    )
    .withColumn(
        "is_delayed",
        F.when(F.col("o.order_status") == "Delayed", F.lit(True)).otherwise(F.lit(False))
    )
    .withColumn(
        "days_to_ship",
        F.datediff(F.col("o.shipping_date"), F.to_date(F.col("o.order_date")))
    )
    .select(
        F.col("o.order_id"),
        F.col("o.order_date"),
        F.col("o.order_calendar_date"),
        F.col("o.order_date_sk").alias("date_sk"),
        F.col("dc.customer_sk"),
        F.col("dp.product_sk"),
        F.col("ds.store_sk"),
        F.col("dch.channel_sk"),
        F.col("o.customer_id"),
        F.col("o.product_id"),
        F.col("o.store_id"),
        F.col("o.channel"),
        F.col("o.quantity"),
        F.col("o.unit_price"),
        F.col("o.discount_amount"),
        F.col("gross_sales_amount"),
        F.col("net_sales_amount"),
        F.col("o.order_status"),
        F.col("o.payment_method"),
        F.col("o.shipping_date"),
        F.col("days_to_ship"),
        F.col("is_cancelled"),
        F.col("is_delayed"),
        F.col("o._source_file_name"),
        F.col("o._ingestion_timestamp"),
        F.col("o._batch_id")
    )
)

write_delta(fact_sales, "gold.fact_sales")

StatementMeta(, b01a3b81-37fc-47f7-9f94-230208af0de3, 8, Finished, Available, Finished, False)

gold.fact_sales: 1206 rows


In [7]:
returns = spark.table("silver.returns").alias("r")
fact_sales_existing = spark.table("gold.fact_sales").alias("fs")
dim_date = spark.table("gold.dim_date").alias("dd")

returns_prepared = (
    returns
    .withColumn("return_calendar_date", F.to_date("return_date"))
    .withColumn("return_date_sk", F.date_format(F.to_date("return_date"), "yyyyMMdd").cast("int"))
)

fact_returns = (
    returns_prepared.alias("r")
    .join(
        fact_sales_existing.alias("fs"),
        F.col("r.order_id") == F.col("fs.order_id"),
        "left"
    )
    .join(
        dim_date.alias("dd"),
        F.col("r.return_date_sk") == F.col("dd.date_sk"),
        "left"
    )
    .select(
        F.col("r.return_id"),
        F.col("r.order_id"),
        F.col("r.return_date"),
        F.col("r.return_calendar_date"),
        F.col("r.return_date_sk").alias("date_sk"),
        F.col("fs.customer_sk"),
        F.col("fs.product_sk"),
        F.col("fs.store_sk"),
        F.col("fs.channel_sk"),
        F.col("r.customer_id"),
        F.col("r.product_id"),
        F.col("r.return_quantity"),
        F.col("r.return_reason"),
        F.col("r.refund_amount"),
        F.col("r.return_status"),
        F.col("fs.net_sales_amount").alias("original_net_sales_amount"),
        F.col("r._source_file_name"),
        F.col("r._ingestion_timestamp"),
        F.col("r._batch_id")
    )
)

write_delta(fact_returns, "gold.fact_returns")

StatementMeta(, b01a3b81-37fc-47f7-9f94-230208af0de3, 9, Finished, Available, Finished, False)

gold.fact_returns: 80 rows


In [8]:
inventory = spark.table("silver.inventory").alias("i")
dim_product = spark.table("gold.dim_product").alias("dp")
dim_store = spark.table("gold.dim_store").alias("ds")
dim_date = spark.table("gold.dim_date").alias("dd")

inventory_prepared = (
    inventory
    .withColumn("snapshot_calendar_date", F.to_date("snapshot_date"))
    .withColumn("snapshot_date_sk", F.date_format(F.to_date("snapshot_date"), "yyyyMMdd").cast("int"))
)

fact_inventory_snapshot = (
    inventory_prepared.alias("i")
    .join(
        dim_product.alias("dp"),
        (
            (F.col("i.product_id") == F.col("dp.product_id")) &
            (F.col("i.snapshot_calendar_date") >= F.col("dp.effective_start_date")) &
            (F.col("i.snapshot_calendar_date") <= F.col("dp.effective_end_date"))
        ),
        "left"
    )
    .join(
        dim_store.alias("ds"),
        F.col("i.store_id") == F.col("ds.store_id"),
        "left"
    )
    .join(
        dim_date.alias("dd"),
        F.col("i.snapshot_date_sk") == F.col("dd.date_sk"),
        "left"
    )
    .withColumn(
        "available_qty",
        F.col("i.on_hand_qty") - F.col("i.reserved_qty")
    )
    .withColumn(
        "is_low_stock",
        F.when(
            (F.col("i.on_hand_qty") - F.col("i.reserved_qty")) <= F.col("i.reorder_level"),
            F.lit(True)
        ).otherwise(F.lit(False))
    )
    .select(
        F.col("i.snapshot_date"),
        F.col("i.snapshot_calendar_date"),
        F.col("i.snapshot_date_sk").alias("date_sk"),
        F.col("dp.product_sk"),
        F.col("ds.store_sk"),
        F.col("i.product_id"),
        F.col("i.store_id"),
        F.col("i.supplier_id"),
        F.col("i.on_hand_qty"),
        F.col("i.reserved_qty"),
        F.col("available_qty"),
        F.col("i.reorder_level"),
        F.col("is_low_stock"),
        F.col("i._source_file_name"),
        F.col("i._ingestion_timestamp"),
        F.col("i._batch_id")
    )
)

write_delta(fact_inventory_snapshot, "gold.fact_inventory_snapshot")

StatementMeta(, b01a3b81-37fc-47f7-9f94-230208af0de3, 10, Finished, Available, Finished, False)

gold.fact_inventory_snapshot: 420 rows


In [9]:
gold_tables = [
    "gold.dim_customer",
    "gold.dim_product",
    "gold.dim_store",
    "gold.dim_channel",
    "gold.dim_date",
    "gold.fact_sales",
    "gold.fact_returns",
    "gold.fact_inventory_snapshot"
]

summary = []

for table_name in gold_tables:
    row_count = spark.table(table_name).count()
    summary.append((table_name, row_count))

summary_df = spark.createDataFrame(summary, ["table_name", "row_count"])
display(summary_df)

StatementMeta(, b01a3b81-37fc-47f7-9f94-230208af0de3, 11, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, bba4d003-14cf-4fb6-9d7f-80ff702c51a5)

In [10]:
display(
    spark.table("gold.fact_sales")
    .select(
        "order_id",
        "order_date",
        "customer_sk",
        "product_sk",
        "store_sk",
        "channel_sk",
        "quantity",
        "gross_sales_amount",
        "net_sales_amount",
        "order_status"
    )
    .limit(20)
)

StatementMeta(, b01a3b81-37fc-47f7-9f94-230208af0de3, 12, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, af2e751e-4b1d-4a52-991b-0f60c70a332d)

In [11]:
display(
    spark.table("gold.fact_inventory_snapshot")
    .filter(F.col("is_low_stock") == True)
    .select(
        "snapshot_date",
        "store_id",
        "product_id",
        "on_hand_qty",
        "reserved_qty",
        "available_qty",
        "reorder_level",
        "is_low_stock"
    )
    .limit(20)
)

StatementMeta(, b01a3b81-37fc-47f7-9f94-230208af0de3, 13, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, d9c86bfc-c3e6-4a1b-8558-d1f40a061b9d)

In [12]:
display(
    spark.table("gold.fact_returns")
    .select(
        "return_id",
        "order_id",
        "return_date",
        "customer_sk",
        "product_sk",
        "return_quantity",
        "refund_amount",
        "return_reason",
        "return_status"
    )
    .limit(20)
)

StatementMeta(, b01a3b81-37fc-47f7-9f94-230208af0de3, 14, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, a922b2f3-08cd-41fd-85e6-815d24d287a1)

In [13]:
display(
    spark.table("gold.fact_sales")
    .groupBy("channel")
    .agg(
        F.countDistinct("order_id").alias("total_orders"),
        F.round(F.sum("net_sales_amount"), 2).alias("total_sales")
    )
    .orderBy(F.desc("total_sales"))
)

StatementMeta(, b01a3b81-37fc-47f7-9f94-230208af0de3, 15, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 6cbcc09f-a4ea-4075-9322-92149095fe81)